# 02 · Sepsis Features

> **Synthetic data only.** Every dataset used in this notebook is synthetic (Synthea, seed 42) or research-use-only (NIH ChestX-ray14). There is no PHI. None of these models is clinically validated or cleared for patient care.

Turn irregular vitals into the model-ready window using the **shared** feature functions in `medflow_ml.features.vitals` — the exact code that runs at serving time. We resample to a 24×5 grid, normalize, and compute rolling slopes.

## Load the shared featurizer
These are the training-time source of truth, mirrored byte-for-byte by `medflow_serving.inference.featurize`.

In [ ]:
from __future__ import annotations
from datetime import datetime, timedelta
import pandas as pd
from medflow_ml.features.vitals import (
    SEQUENCE_STEPS, VITALS_FEATURES, VitalsSample,
    resample_window, normalize_sequence, rolling_stats, slope_per_hour,
)
print('grid steps:', SEQUENCE_STEPS, '| features:', VITALS_FEATURES)

## Build a vitals window
We synthesize a 6-hour deteriorating window (rising HR/RR, falling SpO2) sampled every 30 minutes.

In [ ]:
end = datetime(2026, 6, 11, 12, 0, 0)
samples = []
for k in range(12):  # every 30 min over 6h
    ts = end - timedelta(minutes=30 * (11 - k))
    samples.append(VitalsSample(
        ts=ts,
        heart_rate=80 + 3 * k,
        spo2=98 - 0.4 * k,
        resp_rate=16 + 0.7 * k,
        temp_c=37.0 + 0.1 * k,
        map_mmhg=85 - 0.8 * k,
    ))
len(samples)

## Resample onto the 24×5 grid
Last-observation-carried-forward; leading gaps fall back to population normals.

In [ ]:
grid = resample_window(samples, window_end=end)
frame = pd.DataFrame(grid, columns=list(VITALS_FEATURES))
frame.tail()

## Normalize and plot
Z-scoring against fixed clinical scales centers a normal patient at 0; the rising-HR trend appears as a climbing normalized series.

In [ ]:
import matplotlib.pyplot as plt
norm = normalize_sequence(grid)
norm_df = pd.DataFrame(norm, columns=list(VITALS_FEATURES))
ax = norm_df.plot(figsize=(9, 4), title='Normalized 24-step vitals window')
ax.set_xlabel('step (15-min bins, oldest -> newest)')
ax.set_ylabel('z-score'); plt.tight_layout()

## Rolling slope feature
`rolling_stats` over the window is strictly historical relative to the prediction time — no leakage. A positive heart-rate slope is an early-warning signal.

In [ ]:
points = [(s.ts, s.heart_rate) for s in samples]
stats = rolling_stats(points, end, window_hours=6.0)
print('HR rolling stats (6h):', stats)
print('HR slope per hour     :', round(slope_per_hour(points), 2))

### Takeaways
* The shared featurizer guarantees train/serve parity.
* Slopes summarise trajectories the LSTM also learns from sequence.
* Next: feed these windows to the LSTM (notebook 03).